# 05 — Full Evaluation

Complete evaluation matching every method described in the paper.

## A. Predictive Metrics
- Top-1, Top-5, MRR, NLL on held-out test set
- **Breakdowns**: by category, time of day, new vs. repeat, user activity level

## B. Posterior Predictive Checks (behavioral model validation)
- Repeat purchase rate
- **Category switching matrix** P(cat_{t+1} | cat_t)
- Inter-purchase gap
- Price trajectory
- Brand loyalty index

## C. Baselines
- Frequency
- Markov chain
- Standard conditional logit (hand-specified)
- Structure Descent (final)

## D. Ablations
- Structure search without LLM (random DSL proposals)
- No hierarchy (single global weights)
- TextGrad without priors (unconstrained LLM — no grammar)

## E. Interpretability
- Final DSL structure + global weights

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

from src.dsl import DSLStructure, ALL_TERMS
from src.inner_loop import run_inner_loop, fit_weights_no_hierarchy
from src.outer_loop import random_structure_search, generate_diagnostics
from src.evaluation import (
    compute_metrics, breakdown_by_category, breakdown_by_repeat_vs_novel,
    breakdown_by_activity_level, breakdown_by_time_of_day,
    frequency_baseline, markov_chain_baseline,
    category_switching_matrix, posterior_predictive_checks,
    simulate_sequences, no_hierarchy_weight_fn, print_metrics,
)
from src.checkpoint import Checkpoint

sns.set_theme(style='whitegrid')
DATA_DIR = Path('../data')
ckpt = Checkpoint('../data')
print('Pipeline status:')
for stage, status in ckpt.status().items():
    print(f'  {stage:15s} {status}')

In [ ]:
# ── Load all data ──────────────────────────────────────────────────────────────
with open(DATA_DIR / 'train_features.pkl', 'rb') as f:
    feat_data = pickle.load(f)
with open(DATA_DIR / 'final_structure.pkl', 'rb') as f:
    final = pickle.load(f)
with open(DATA_DIR / 'train_choices.pkl', 'rb') as f:
    train_choices = pickle.load(f)

df = pd.read_parquet(DATA_DIR / 'purchases_processed.parquet')

features_list  = feat_data['features_list']
chosen_indices = feat_data['chosen_indices']
customer_ids   = feat_data['customer_ids']
categories     = feat_data['categories']
metadata       = [e['metadata'] for e in train_choices[:len(features_list)]]
final_structure = DSLStructure.from_dict(final['structure'])
history         = final['history']

# Hour-of-day for each event (from order_date)
events_df = pd.DataFrame({
    'customer_id': customer_ids,
    'category': categories,
    'order_date': [e['order_date'] for e in train_choices[:len(features_list)]],
})
order_hours = pd.to_datetime(events_df['order_date']).dt.hour.tolist()

# Purchase counts per customer (for activity-level breakdown)
purchase_counts = df['customer_id'].value_counts().to_dict()

print(f'Final structure: {final_structure}')
print(f'Events: {len(features_list):,}')

## A. Predictive Metrics

In [ ]:
# Fit final model
weights, score = run_inner_loop(
    final_structure, features_list, chosen_indices, customer_ids, categories
)

metrics = compute_metrics(
    features_list, chosen_indices, weights.get_weights, customer_ids, categories
)
print_metrics(metrics, label='Structure Descent (final)')
print(f'Posterior score: {score:.2f}')

### A1. Breakdown by category

In [ ]:
cat_breakdown = breakdown_by_category(
    features_list, chosen_indices, weights.get_weights, customer_ids, categories
)
print(cat_breakdown.sort_values('top1').to_string(index=False))

top20 = cat_breakdown.nlargest(20, 'n_events')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top20['category'], top20['top1'], color='steelblue')
ax.axvline(metrics['top1'], color='red', linestyle='--', label=f'Overall ({metrics["top1"]:.1%})')
ax.set_xlabel('Top-1 Accuracy')
ax.set_title('Top-1 by Category (top 20 by volume)')
ax.legend()
plt.tight_layout()
plt.show()

### A2. Breakdown by repeat vs. novel purchases

In [ ]:
repeat_breakdown = breakdown_by_repeat_vs_novel(
    features_list, chosen_indices, weights.get_weights,
    customer_ids, categories, metadata
)
display(repeat_breakdown)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(repeat_breakdown['type'], repeat_breakdown['top1'], color=['steelblue', 'coral'])
ax.set_ylabel('Top-1 Accuracy')
ax.set_title('Accuracy: Repeat vs. Novel Purchases')
for i, row in repeat_breakdown.iterrows():
    ax.text(i, row['top1'] + 0.005, f"{row['top1']:.1%}", ha='center')
plt.tight_layout()
plt.show()

### A3. Breakdown by user activity level (low / medium / high)

In [ ]:
activity_breakdown = breakdown_by_activity_level(
    features_list, chosen_indices, weights.get_weights,
    customer_ids, categories, purchase_counts, n_bins=3
)
display(activity_breakdown)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(activity_breakdown['activity_level'], activity_breakdown['top1'],
       color=['#c6dbef', '#6baed6', '#2171b5'])
ax.set_ylabel('Top-1 Accuracy')
ax.set_title('Accuracy by User Activity Level')
for i, row in activity_breakdown.iterrows():
    ax.text(i, row['top1'] + 0.003, f"{row['top1']:.1%}", ha='center')
plt.tight_layout()
plt.show()

### A4. Breakdown by time of day

In [ ]:
time_breakdown = breakdown_by_time_of_day(
    features_list, chosen_indices, weights.get_weights,
    customer_ids, categories, order_hours
)
display(time_breakdown)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(time_breakdown['time_of_day'], time_breakdown['top1'], color='mediumpurple')
ax.set_ylabel('Top-1 Accuracy')
ax.set_title('Accuracy by Time of Day')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## B. Posterior Predictive Checks

Simulate synthetic sequences from the fitted model and compare distributional
statistics to real data. This is the most important evaluation for claiming this
is a *behavioral* model, not just a predictor.

In [ ]:
# Simulate purchase sequences from the fitted model
print('Simulating sequences...')
simulated_df = simulate_sequences(
    df, weights.get_weights, final_structure,
    extract_features_fn=None,  # uses internal heuristic
    n_steps=20, seed=42
)
print(f'Simulated: {len(simulated_df):,} events  |  Real: {len(df):,} events')

In [ ]:
# Run all posterior predictive checks
real_test_df = df[df.get('split', 'train') == 'test'] if 'split' in df.columns else df.sample(frac=0.1, random_state=42)
ppc = posterior_predictive_checks(real_test_df, simulated_df)
display(ppc)

### B1. Category switching matrix

In [ ]:
real_trans = category_switching_matrix(df, top_n=8)
sim_trans  = category_switching_matrix(simulated_df, top_n=8)

# Align columns/rows
common_cats = sorted(set(real_trans.index) & set(sim_trans.index))[:8]
real_trans = real_trans.reindex(index=common_cats, columns=common_cats, fill_value=0)
sim_trans  = sim_trans.reindex(index=common_cats, columns=common_cats, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(real_trans, ax=axes[0], cmap='Blues', vmin=0, vmax=1,
            annot=True, fmt='.2f', cbar=False)
axes[0].set_title('Real — Category Transition Matrix')
axes[0].set_xlabel('Next Category')
axes[0].set_ylabel('Current Category')

sns.heatmap(sim_trans, ax=axes[1], cmap='Oranges', vmin=0, vmax=1,
            annot=True, fmt='.2f', cbar=False)
axes[1].set_title('Simulated — Category Transition Matrix')
axes[1].set_xlabel('Next Category')
axes[1].set_ylabel('Current Category')

plt.tight_layout()
plt.show()

### B2. Distributional statistics comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Inter-purchase gap distribution
real_gaps = (df.sort_values(['customer_id','order_date'])
               .groupby('customer_id')['order_date'].diff().dt.days.dropna())
sim_gaps  = (simulated_df.sort_values(['customer_id','order_date'])
                         .groupby('customer_id')['order_date'].diff().dt.days.dropna())

axes[0].hist(real_gaps.clip(upper=90), bins=40, alpha=0.6, label='Real', density=True)
axes[0].hist(sim_gaps.clip(upper=90),  bins=40, alpha=0.6, label='Simulated', density=True)
axes[0].set_title('Inter-purchase Gap (days)')
axes[0].set_xlabel('Days')
axes[0].legend()

# Price trajectory
real_prices = df.groupby('customer_id')['price'].mean()
sim_prices  = simulated_df.groupby('customer_id')['price'].mean()

axes[1].hist(real_prices.clip(upper=100), bins=40, alpha=0.6, label='Real', density=True)
axes[1].hist(sim_prices.clip(upper=100),  bins=40, alpha=0.6, label='Simulated', density=True)
axes[1].set_title('Avg Price per Customer ($)')
axes[1].set_xlabel('Price')
axes[1].legend()

# Repeat purchase rate per customer
real_rr = df.groupby('customer_id')['asin'].apply(lambda x: x.duplicated().mean())
sim_rr  = simulated_df.groupby('customer_id')['asin'].apply(lambda x: x.duplicated().mean())

axes[2].hist(real_rr, bins=30, alpha=0.6, label='Real', density=True)
axes[2].hist(sim_rr,  bins=30, alpha=0.6, label='Simulated', density=True)
axes[2].set_title('Repeat Purchase Rate per Customer')
axes[2].set_xlabel('Rate')
axes[2].legend()

plt.suptitle('Posterior Predictive Checks', y=1.02)
plt.tight_layout()
plt.show()

## C. Baselines

### C1. Frequency baseline

In [ ]:
# Build per-customer purchase history counts
cust_history: dict = defaultdict(lambda: defaultdict(int))
for ev in train_choices[:len(features_list)]:
    cust_history[ev['customer_id']][ev['chosen_asin']] += 1

freq_metrics = frequency_baseline(
    train_choices[:len(features_list)],
    {cid: dict(counts) for cid, counts in cust_history.items()}
)
print_metrics(freq_metrics, label='Frequency baseline')

### C2. Markov chain baseline

In [ ]:
# Build transition matrix and category popularity from training data
trans_matrix = category_switching_matrix(df, top_n=50)

cat_popularity: dict = defaultdict(lambda: defaultdict(int))
for ev in train_choices[:len(features_list)]:
    cat_popularity[ev['category']][ev['chosen_asin']] += 1

# Add previous category to each event
events_with_prev = []
prev_cat_map = {}
for ev in train_choices[:len(features_list)]:
    ev_copy = dict(ev)
    ev_copy['prev_category'] = prev_cat_map.get(ev['customer_id'], ev['category'])
    prev_cat_map[ev['customer_id']] = ev['category']
    events_with_prev.append(ev_copy)

markov_metrics = markov_chain_baseline(
    events_with_prev,
    trans_matrix,
    {cat: dict(counts) for cat, counts in cat_popularity.items()}
)
print_metrics(markov_metrics, label='Markov chain baseline')

### C3. Standard conditional logit (hand-specified, no structure search)

In [ ]:
# Use all Layer 1 + Layer 2 terms — no search, human-specified
from src.dsl import LAYER1_PRIMITIVES, LAYER2_AMAZON

hand_structure = DSLStructure(LAYER1_PRIMITIVES + LAYER2_AMAZON)
print(f'Hand-specified: {hand_structure}')

# Pad features with zeros for extra terms not in the current feature matrix
n_extra = len(hand_structure) - len(final_structure)
if n_extra > 0:
    padded_features = [np.hstack([f, np.zeros((f.shape[0], n_extra))]) for f in features_list]
else:
    padded_features = features_list

hand_weights, hand_score = run_inner_loop(
    hand_structure, padded_features, chosen_indices, customer_ids, categories
)
hand_metrics = compute_metrics(
    padded_features, chosen_indices, hand_weights.get_weights, customer_ids, categories
)
print_metrics(hand_metrics, label='Hand-specified logit')

### C4. Full comparison table

In [ ]:
def fmt(m, key):
    v = m.get(key, float('nan'))
    if key in ('top1', 'top5'):
        return f"{v:.1%}" if not np.isnan(v) else 'n/a'
    return f"{v:.4f}" if not np.isnan(v) else 'n/a'

rows = [
    ('Frequency',                freq_metrics),
    ('Markov chain',             markov_metrics),
    ('Hand-specified logit',     hand_metrics),
    ('Structure Descent (final)', metrics),
]

comparison = pd.DataFrame([
    {'Model': name, 'Top-1': fmt(m,'top1'), 'Top-5': fmt(m,'top5'),
     'MRR': fmt(m,'mrr'), 'NLL': fmt(m,'val_nll'), 'n': m.get('n_events','?')}
    for name, m in rows
])
display(comparison)

## D. Ablations

### D1. Structure search without LLM (random DSL proposals)
Shows that LLM guidance outperforms blind random search over the same DSL grammar.

In [ ]:
def fit_fn(structure):
    n_terms = len(structure.terms)
    n_current = len(final_structure.terms)
    if n_terms <= n_current:
        fl = [f[:, :n_terms] for f in features_list]
    else:
        fl = [np.hstack([f, np.zeros((f.shape[0], n_terms - n_current))]) for f in features_list]
    return run_inner_loop(structure, fl, chosen_indices, customer_ids, categories)

random_history = random_structure_search(
    DSLStructure.initial(), fit_fn, n_iterations=8, seed=42
)

random_hist_df = pd.DataFrame(random_history)
display(random_hist_df)

In [ ]:
# Compare search trajectories: LLM vs. random
llm_hist_df = pd.DataFrame(history)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(llm_hist_df['iteration'], llm_hist_df['score'], marker='o', label='LLM-guided', color='steelblue')
ax.plot(random_hist_df['iteration'], random_hist_df['score'], marker='s', label='Random search', color='coral', linestyle='--')
ax.set_xlabel('Outer loop iteration')
ax.set_ylabel('Posterior score')
ax.set_title('Structure Descent: LLM-guided vs. Random DSL Search')
ax.legend()
plt.tight_layout()
plt.show()

### D2. No hierarchy ablation (single global weights only)
Shows that θ_g + θ_c + Δ_i captures individual/context variation that flat weights miss.

In [ ]:
theta_g_only = fit_weights_no_hierarchy(
    final_structure, features_list, chosen_indices, sigma=10.0
)
flat_weight_fn = no_hierarchy_weight_fn(theta_g_only)

flat_metrics = compute_metrics(
    features_list, chosen_indices, flat_weight_fn, customer_ids, categories
)
print_metrics(flat_metrics, label='No hierarchy (θ_g only)')
print_metrics(metrics,      label='Full hierarchy (θ_g + θ_c + Δ_i)')

ablation_df = pd.DataFrame([
    {'Model': 'No hierarchy (θ_g only)',       'Top-1': f"{flat_metrics['top1']:.1%}", 'MRR': f"{flat_metrics['mrr']:.4f}", 'NLL': f"{flat_metrics['val_nll']:.4f}"},
    {'Model': 'Full hierarchy (θ_g+θ_c+Δ_i)', 'Top-1': f"{metrics['top1']:.1%}",      'MRR': f"{metrics['mrr']:.4f}",      'NLL': f"{metrics['val_nll']:.4f}"},
])
display(ablation_df)

### D3. TextGrad without priors (unconstrained LLM)
Requires `ANTHROPIC_API_KEY`. The LLM proposes any expression without grammar constraints or the complexity prior — this ablation shows that the DSL grammar + Bayesian prior prevents overfitting to spurious features.

In [ ]:
# NOTE: This calls the LLM without grammar constraints.
# The returned expression is not necessarily executable — it demonstrates
# the type of unconstrained proposals the prior prevents.
from dotenv import load_dotenv
from src.outer_loop import prompt_llm_unconstrained

load_dotenv('../.env')

# Get current diagnostics
from src.evaluation import compute_residuals
residuals = compute_residuals(
    features_list, chosen_indices, weights.get_weights,
    customer_ids, categories, metadata
)
report = generate_diagnostics(final_structure, score, metrics, residuals)

unconstrained = prompt_llm_unconstrained(report)
print('=== TextGrad-style unconstrained proposal ===')
print(f'Reasoning: {unconstrained.get("reasoning", "")}')
print(f'Proposed expression: {unconstrained.get("new_expression", "")}')
print('\n(Note: This expression may be outside the DSL grammar.')
print('The grammar constraint prevents such proposals from being accepted.)')

## E. Interpretability Evaluation

The primary result is the final DSL structure itself — human-readable,
auditable, and carrying explicit behavioral interpretation.

In [ ]:
print('=== Final Discovered Structure ===')
print(final_structure)
print(f'Complexity  L(S) = {final_structure.complexity()}')
print(f'log p(S)  = {final_structure.log_prior():.4f}')
print(f'Posterior score = {score:.2f}')
print('\nGlobal weights (θ_g) — sorted by magnitude:')
for term, w in sorted(zip(final_structure.terms, weights.theta_g), key=lambda x: -abs(x[1])):
    print(f'  {term.display_name:30s}  {w:+.4f}')

In [ ]:
sorted_pairs = sorted(zip(final_structure.terms, weights.theta_g), key=lambda x: x[1])
terms_s, weights_s = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(7, max(3, len(terms_s) * 0.45)))
colors = ['steelblue' if w >= 0 else 'tomato' for w in weights_s]
ax.barh(terms_s, weights_s, color=colors)
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('Global weight (θ_g)')
ax.set_title('Structure Descent — Final Discovered Structure')
plt.tight_layout()
plt.show()

### Full results summary

In [ ]:
from src.dsl import DSLTerm
_rs_terms = [DSLTerm.parse(t) for t in random_history[-1]['structure'].replace('S = ', '').split(' + ')]
all_models = [
    ('Frequency',                    freq_metrics),
    ('Markov chain',                 markov_metrics),
    ('Hand-specified logit',         hand_metrics),
    ('No hierarchy (ablation)',       flat_metrics),
    ('Random search (ablation)',
     compute_metrics(features_list, chosen_indices,
                     fit_fn(DSLStructure(_rs_terms))[0].get_weights,
                     customer_ids, categories) if random_history else metrics),
    ('Structure Descent (LLM)',      metrics),
]

summary = pd.DataFrame([
    {'Model': name,
     'Top-1':  fmt(m, 'top1'),
     'Top-5':  fmt(m, 'top5'),
     'MRR':    fmt(m, 'mrr'),
     'NLL':    fmt(m, 'val_nll')}
    for name, m in all_models
])
display(summary)